In [0]:
pip install nflreadpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 133.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import nflreadpy as nfl
pbp_reg = nfl.load_pbp(range(2015, 2026))

In [0]:
pbp.columns

['play_id',
 'game_id',
 'old_game_id',
 'home_team',
 'away_team',
 'season_type',
 'week',
 'posteam',
 'posteam_type',
 'defteam',
 'side_of_field',
 'yardline_100',
 'game_date',
 'quarter_seconds_remaining',
 'half_seconds_remaining',
 'game_seconds_remaining',
 'game_half',
 'quarter_end',
 'drive',
 'sp',
 'qtr',
 'down',
 'goal_to_go',
 'time',
 'yrdln',
 'ydstogo',
 'ydsnet',
 'desc',
 'play_type',
 'yards_gained',
 'shotgun',
 'no_huddle',
 'qb_dropback',
 'qb_kneel',
 'qb_spike',
 'qb_scramble',
 'pass_length',
 'pass_location',
 'air_yards',
 'yards_after_catch',
 'run_location',
 'run_gap',
 'field_goal_result',
 'kick_distance',
 'extra_point_result',
 'two_point_conv_result',
 'home_timeouts_remaining',
 'away_timeouts_remaining',
 'timeout',
 'timeout_team',
 'td_team',
 'td_player_name',
 'td_player_id',
 'posteam_timeouts_remaining',
 'defteam_timeouts_remaining',
 'total_home_score',
 'total_away_score',
 'posteam_score',
 'defteam_score',
 'score_differential',
 'po

In [0]:
import polars as pl
pbp_reg = pbp.filter(pl.col("season_type") == "REG")

In [0]:
# ── Passing
passing = (
    pbp_reg.filter(pl.col("passer_player_id").is_not_null())
    .group_by("passer_player_id", "passer_player_name", "season")
    .agg(
        (pl.col("complete_pass").sum() + pl.col("incomplete_pass").sum()).alias("pass_attempts"),
        pl.col("complete_pass").sum().alias("completions"),
        pl.col("passing_yards").sum().alias("pass_yards"),
        pl.col("pass_touchdown").sum().alias("pass_tds"),
        pl.col("interception").sum().alias("interceptions"),
        pl.col("sack").sum().alias("sacks"),
        pl.col("epa").mean().alias("pass_epa_per_play"),
        pl.col("epa").sum().alias("pass_total_epa"),
    )
    .with_columns([
        (pl.col("completions") / pl.col("pass_attempts")).alias("completion_pct"),
        (pl.col("pass_yards") / pl.col("pass_attempts")).alias("yards_per_attempt"),
        (pl.col("pass_tds") / pl.col("pass_attempts")).alias("pass_td_rate"),
        (pl.col("interceptions") / pl.col("pass_attempts")).alias("int_rate"),
    ])
    .rename({"passer_player_id": "player_id", "passer_player_name": "player_name"})
)

In [0]:
# ── Rushing
rushing = (
    pbp_reg.filter(pl.col("rusher_player_id").is_not_null())
    .group_by("rusher_player_id", "rusher_player_name", "season")
    .agg(
        pl.col("rush_attempt").sum().alias("carries"),
        pl.col("rushing_yards").sum().alias("rush_yards"),
        pl.col("rush_touchdown").sum().alias("rush_tds"),
        (pl.col("rushing_yards") >= 10).sum().alias("explosive_runs"),
        pl.col("fumble").sum().alias("rush_fumbles"),
        pl.col("epa").mean().alias("rush_epa_per_play"),
        pl.col("epa").sum().alias("rush_total_epa"),
    )
    .with_columns([
        (pl.col("rush_yards") / pl.col("carries")).alias("yards_per_carry"),
        (pl.col("rush_tds") / pl.col("carries")).alias("rush_td_rate"),
        (pl.col("explosive_runs") / pl.col("carries")).alias("explosive_run_rate"),
        (pl.col("rush_fumbles") / pl.col("carries")).alias("fumble_rate"),
    ])
    .rename({"rusher_player_id": "player_id", "rusher_player_name": "player_name"})
)

In [0]:
# ── Receiving 
receiving = (
    pbp_reg.filter(pl.col("receiver_player_id").is_not_null())
    .group_by("receiver_player_id", "receiver_player_name", "season")
    .agg(
        pl.col("pass_attempt").sum().alias("targets"),
        pl.col("complete_pass").sum().alias("receptions"),
        pl.col("receiving_yards").sum().alias("rec_yards"),
        pl.col("pass_touchdown").sum().alias("rec_tds"),
        (pl.col("receiving_yards") >= 20).sum().alias("explosive_recs"),
        pl.col("fumble").sum().alias("rec_fumbles"),
        pl.col("air_yards").sum().alias("air_yards"),
        pl.col("yards_after_catch").sum().alias("yac"),
        pl.col("epa").mean().alias("rec_epa_per_play"),
        pl.col("epa").sum().alias("rec_total_epa"),
    )
    .with_columns([
        (pl.col("receptions") / pl.col("targets")).alias("catch_rate"),
        (pl.col("rec_yards") / pl.col("receptions")).alias("yards_per_rec"),
        (pl.col("rec_tds") / pl.col("targets")).alias("rec_td_rate"),
        (pl.col("explosive_recs") / pl.col("receptions")).alias("explosive_rec_rate"),
        (pl.col("air_yards") / pl.col("targets")).alias("air_yards_per_target"),
        (pl.col("yac") / pl.col("receptions")).alias("yac_per_rec"),
    ])
    .rename({"receiver_player_id": "player_id", "receiver_player_name": "player_name"})
)

In [0]:
# Compute games_played once across all play types
games_played = pl.concat([
    pbp_reg.filter(pl.col("passer_player_id").is_not_null())
        .select(pl.col("passer_player_id").alias("player_id"), "season", "game_id"),
    pbp_reg.filter(pl.col("rusher_player_id").is_not_null())
        .select(pl.col("rusher_player_id").alias("player_id"), "season", "game_id"),
    pbp_reg.filter(pl.col("receiver_player_id").is_not_null())
        .select(pl.col("receiver_player_id").alias("player_id"), "season", "game_id"),
]).group_by("player_id", "season").agg(
    pl.col("game_id").n_unique().alias("games_played")
)

In [0]:
# ── Combine into one row per player per season 
# Start from the union of all player/season combos
all_players = pl.concat([
    passing.select("player_id", "player_name", "season"),
    rushing.select("player_id", "player_name", "season"),
    receiving.select("player_id", "player_name", "season"),
]).unique()

player_season = (
    all_players
    .join(passing.drop("player_name"),   on=["player_id", "season"], how="left")
    .join(rushing.drop("player_name"),   on=["player_id", "season"], how="left")
    .join(receiving.drop("player_name"), on=["player_id", "season"], how="left")
    .join(games_played, on=["player_id", "season"], how="left")
    .fill_null(0)
)

In [0]:
contracts = nfl.load_contracts()
contracts.columns

['player',
 'position',
 'team',
 'is_active',
 'year_signed',
 'years',
 'value',
 'apy',
 'guaranteed',
 'apy_cap_pct',
 'inflated_value',
 'inflated_apy',
 'inflated_guaranteed',
 'player_page',
 'otc_id',
 'gsis_id',
 'date_of_birth',
 'height',
 'weight',
 'college',
 'draft_year',
 'draft_round',
 'draft_overall',
 'draft_team',
 'cols']

In [0]:
contracts_clean = contracts.filter(
    pl.col("gsis_id").is_not_null()
).select([
    "gsis_id",
    "player",
    "position",
    "year_signed",
    "years",
    "value",
    "apy",
    "guaranteed",
    "is_active",
    "apy_cap_pct",
    "inflated_value",
    "inflated_apy",
    "inflated_guaranteed",
    "draft_round",
    "draft_overall"
])

In [0]:
contracts_clean = contracts_clean.with_columns([
    (pl.col("year_signed") + pl.col("years") - 1).alias("year_end")
])
contracts_clean.head()

gsis_id,player,position,year_signed,years,value,apy,guaranteed,is_active,apy_cap_pct,inflated_value,inflated_apy,inflated_guaranteed,draft_round,draft_overall,year_end
str,str,str,i32,i32,f64,f64,f64,bool,f64,f64,f64,f64,i32,i32,i32
"""00-0036442""","""Joe Burrow""","""QB""",2023,5,275.0,55.0,146.51,true,0.245,368.460854,73.692171,196.302544,1,1,2027
"""00-0023459""","""Aaron Rodgers""","""QB""",2022,5,150.815,50.271667,101.415,false,0.241,218.181931,72.727311,146.715648,1,24,2026
"""00-0034857""","""Josh Allen""","""QB""",2021,6,258.0,43.0,100.0,false,0.236,425.806027,70.967671,165.041096,1,7,2026
"""00-0029263""","""Russell Wilson""","""QB""",2022,5,245.0,49.0,124.0,false,0.235,354.43804,70.887608,179.389049,3,75,2026
"""00-0033077""","""Dak Prescott""","""QB""",2024,4,240.0,60.0,129.0,true,0.235,283.038371,70.759593,152.133125,4,135,2027


In [0]:
contracts_clean = contracts_clean.sort(["gsis_id", "year_signed"], descending=[False, True])

In [0]:
next_contracts = contracts_clean.select([
    "gsis_id",
    "player",
    "year_signed",
    "apy",
    "years",
    "value",
    "guaranteed",
    "is_active",
    "apy_cap_pct",
    "inflated_value",
    "inflated_apy",
    "inflated_guaranteed",
    "draft_round",
    "draft_overall"
]).rename({
    "year_signed": "next_year_signed",
    "apy": "next_apy",
    "years": "next_years",
    "value": "next_value",
    "guaranteed": "next_guaranteed",
    "apy_cap_pct": "next_apy_cap_pct",
    "inflated_value": "next_inflated_value",
    "inflated_apy": "next_inflated_apy",
    "inflated_guaranteed": "next_inflated_guaranteed",
})
next_contracts.head()

gsis_id,player,next_year_signed,next_apy,next_years,next_value,next_guaranteed,is_active,next_apy_cap_pct,next_inflated_value,next_inflated_apy,next_inflated_guaranteed,draft_round,draft_overall
str,str,i32,f64,i32,f64,f64,bool,f64,f64,f64,f64,i32,i32
"""00-0000108""","""David Akers""",0,1.005,1,1.005,0.1,false,0.0,0.0,0.0,0.0,null,null
"""00-0000585""","""Champ Bailey""",2014,1.875,2,3.75,0.5,false,0.014,8.492481,4.246241,1.132331,1,7
"""00-0000585""","""Champ Bailey""",2011,10.75,4,43.0,10.5,false,0.089,107.593769,26.898442,26.272897,1,7
"""00-0000585""","""Champ Bailey""",2004,6.801,1,6.801,6.801,false,0.084,25.420828,25.420828,25.420828,1,7
"""00-0000585""","""Champ Bailey""",2004,9.004575,7,63.032025,6.0,false,0.112,235.601573,33.657368,22.426845,1,7


In [0]:
next_contracts_clean = (
    next_contracts
    .sort(["gsis_id", "next_year_signed", "next_apy"])
    .group_by(["gsis_id", "next_year_signed"])
    .agg(pl.all().last())
)

In [0]:
# 1. join all contracts to all seasons
joined = player_season.join(
    next_contracts_clean,
    left_on="player_id",
    right_on="gsis_id",
    how="left"
)

# 2. keep only future contracts
future = joined.filter(
    pl.col("next_year_signed").is_null() | 
    (pl.col("next_year_signed") > pl.col("season"))
)

# 3. pick closest future contract
final = (
    future
    .sort(["player_id", "season", "next_year_signed"])
    .group_by(["player_id", "season"])
    .agg(pl.all().first())
)

In [0]:
# season → final contract in that season → next contract in a later season

In [0]:
current_contracts = contracts_clean.rename({
    "year_signed": "curr_year_signed",
    "apy": "curr_apy",
    "years": "curr_years",
    "value": "curr_value",
    "guaranteed": "curr_guaranteed",
    "apy_cap_pct": "curr_apy_cap_pct",
    "inflated_value": "curr_inflated_value",
    "inflated_apy": "curr_inflated_apy",
    "inflated_guaranteed": "curr_inflated_guaranteed",
})

In [0]:
season_features = (
    player_season.join(
        current_contracts,
        left_on="player_id",
        right_on="gsis_id",
        how="left"
    )
    .filter(
        (pl.col("season") >= pl.col("curr_year_signed")) &
        (pl.col("season") <= pl.col("year_end"))
    )
    .sort(["player_id", "season", "curr_year_signed"])
    .unique(subset=["player_id", "season"], keep="last")
)

In [0]:
season_features

player_id,player_name,season,pass_attempts,completions,pass_yards,pass_tds,interceptions,sacks,pass_epa_per_play,pass_total_epa,completion_pct,yards_per_attempt,pass_td_rate,int_rate,carries,rush_yards,rush_tds,explosive_runs,rush_fumbles,rush_epa_per_play,rush_total_epa,yards_per_carry,rush_td_rate,explosive_run_rate,fumble_rate,targets,receptions,rec_yards,rec_tds,explosive_recs,rec_fumbles,air_yards,yac,rec_epa_per_play,rec_total_epa,catch_rate,yards_per_rec,rec_td_rate,explosive_rec_rate,air_yards_per_target,yac_per_rec,games_played,player,position,curr_year_signed,curr_years,curr_value,curr_apy,curr_guaranteed,is_active,curr_apy_cap_pct,curr_inflated_value,curr_inflated_apy,curr_inflated_guaranteed,draft_round,draft_overall,year_end
str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,str,str,i32,i32,f64,f64,f64,bool,f64,f64,f64,f64,i32,i32,i32
"""00-0027003""","""D.Brown""",2015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59.0,229.0,1.0,6,0.0,-0.132814,-7.836046,3.881356,0.016949,0.101695,0.0,13.0,8.0,88.0,0.0,2,1.0,-36.0,110.0,0.199691,2.595978,0.615385,11.0,0.0,0.25,-2.769231,13.75,6,"""Donald Brown""","""RB""",2015,2,6.5,3.25,0.0,false,0.023,13.664154,6.832077,0.0,1,27,2016
"""00-0035287""","""K.Johnson""",2019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0,0.0,-0.647963,-0.647963,3.0,0.0,0.0,0.0,43.0,21.0,187.0,1.0,1,1.0,394.0,43.0,-0.149294,-6.419639,0.488372,8.904762,0.023256,0.047619,9.162791,2.047619,10,"""KeeSean Johnson""","""WR""",2019,4,2.729312,0.682328,0.209312,false,0.004,4.368059,1.092015,0.334988,6,174,2022
"""00-0030506""","""T.Kelce""",2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,121.0,93.0,984.0,5.0,12,2.0,807.0,470.0,0.396818,48.015015,0.768595,10.580645,0.041322,0.129032,6.669421,5.053763,15,"""Travis Kelce""","""TE""",2020,4,57.25,14.3125,21.0,false,0.072,87.001514,21.750378,31.913219,3,63,2023
"""00-0037253""","""Ma.Jones""",2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0,78.0,1.0,1,0.0,-17.0,95.0,1.594512,6.378046,1.0,19.5,0.25,0.25,-4.25,23.75,3,"""Marcus Jones""","""CB""",2022,4,5.176952,1.294238,0.945056,false,0.006,7.489423,1.872356,1.367199,3,85,2025
"""00-0033006""","""J.Mickens""",2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,7.0,58.0,0.0,0,0.0,73.0,31.0,0.459322,4.593218,0.7,8.285714,0.0,0.0,7.3,4.428571,2,"""Jaydon Mickens""","""WR""",2020,2,1.67,0.835,0.0,false,0.004,2.537861,1.26893,0.0,null,null,2021
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""00-0035645""","""G.Olszewski""",2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,9.0,0.0,0,0.0,0.76596,0.76596,9.0,0.0,0.0,0.0,3.0,2.0,31.0,0.0,1,0.0,29.0,14.0,0.559231,1.677693,0.666667,15.5,0.0,0.5,9.666667,7.0,4,"""Gunner Olszewski""","""WR""",2019,3,1.7575,0.585833,0.0025,false,0.003,2.812747,0.937582,0.004001,null,null,2021
"""00-0030569""","""S.Renfree""",2015,6.0,3.0,11.0,0.0,1.0,2.0,-0.824322,-7.4189,0.5,1.833333,0.0,0.166667,2.0,-4.0,0.0,0,0.0,-0.967829,-1.935659,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,"""Sean Renfree""","""QB""",2013,4,2.205896,0.551474,0.045896,false,0.004,5.401755,1.350439,0.112389,7,249,2016
"""00-0032160""","""T.Williams""",2015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,2.0,90.0,1.0,1,0.0,161.0,51.0,-0.102325,-0.613951,0.333333,45.0,0.166667,0.5,26.833333,25.5,2,"""Tyrell Williams""","""WR""",2015,3,1.575,0.525,0.0,false,0.004,3.31093,1.103643,0.0,null,null,2017


In [0]:
dataset = season_features.join(
    final,
    on=["player_id", "season"],
    how="left"
)

In [0]:
dataset = dataset.select(
    [col for col in dataset.columns if not col.endswith("_right")]
)
dataset

player_id,player_name,season,pass_attempts,completions,pass_yards,pass_tds,interceptions,sacks,pass_epa_per_play,pass_total_epa,completion_pct,yards_per_attempt,pass_td_rate,int_rate,carries,rush_yards,rush_tds,explosive_runs,rush_fumbles,rush_epa_per_play,rush_total_epa,yards_per_carry,rush_td_rate,explosive_run_rate,fumble_rate,targets,receptions,rec_yards,rec_tds,explosive_recs,rec_fumbles,air_yards,yac,rec_epa_per_play,rec_total_epa,catch_rate,yards_per_rec,rec_td_rate,explosive_rec_rate,air_yards_per_target,yac_per_rec,games_played,player,position,curr_year_signed,curr_years,curr_value,curr_apy,curr_guaranteed,is_active,curr_apy_cap_pct,curr_inflated_value,curr_inflated_apy,curr_inflated_guaranteed,draft_round,draft_overall,year_end,next_year_signed,next_apy,next_years,next_value,next_guaranteed,next_apy_cap_pct,next_inflated_value,next_inflated_apy,next_inflated_guaranteed
str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,str,str,i32,i32,f64,f64,f64,bool,f64,f64,f64,f64,i32,i32,i32,i32,f64,i32,f64,f64,f64,f64,f64,f64
"""00-0027003""","""D.Brown""",2015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59.0,229.0,1.0,6,0.0,-0.132814,-7.836046,3.881356,0.016949,0.101695,0.0,13.0,8.0,88.0,0.0,2,1.0,-36.0,110.0,0.199691,2.595978,0.615385,11.0,0.0,0.25,-2.769231,13.75,6,"""Donald Brown""","""RB""",2015,2,6.5,3.25,0.0,false,0.023,13.664154,6.832077,0.0,1,27,2016,2016,0.965,1,0.965,0.3,0.006,1.871952,1.871952,0.581954
"""00-0035287""","""K.Johnson""",2019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0,0.0,-0.647963,-0.647963,3.0,0.0,0.0,0.0,43.0,21.0,187.0,1.0,1,1.0,394.0,43.0,-0.149294,-6.419639,0.488372,8.904762,0.023256,0.047619,9.162791,2.047619,10,"""KeeSean Johnson""","""WR""",2019,4,2.729312,0.682328,0.209312,false,0.004,4.368059,1.092015,0.334988,6,174,2022,2021,0.1656,1,0.1656,0.0,0.001,0.273308,0.273308,0.0
"""00-0030506""","""T.Kelce""",2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,121.0,93.0,984.0,5.0,12,2.0,807.0,470.0,0.396818,48.015015,0.768595,10.580645,0.041322,0.129032,6.669421,5.053763,15,"""Travis Kelce""","""TE""",2020,4,57.25,14.3125,21.0,false,0.072,87.001514,21.750378,31.913219,3,63,2023,2024,17.125,2,34.25,17.0,0.067,40.391934,20.195967,20.048551
"""00-0037253""","""Ma.Jones""",2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0,78.0,1.0,1,0.0,-17.0,95.0,1.594512,6.378046,1.0,19.5,0.25,0.25,-4.25,23.75,3,"""Marcus Jones""","""CB""",2022,4,5.176952,1.294238,0.945056,false,0.006,7.489423,1.872356,1.367199,3,85,2025,2025,12.0,3,36.0,17.5,0.043,38.836676,12.945559,18.87894
"""00-0033006""","""J.Mickens""",2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,7.0,58.0,0.0,0,0.0,73.0,31.0,0.459322,4.593218,0.7,8.285714,0.0,0.0,7.3,4.428571,2,"""Jaydon Mickens""","""WR""",2020,2,1.67,0.835,0.0,false,0.004,2.537861,1.26893,0.0,null,null,2021,2021,0.99,1,0.99,0.0,0.005,1.633907,1.633907,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""00-0035645""","""G.Olszewski""",2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,9.0,0.0,0,0.0,0.76596,0.76596,9.0,0.0,0.0,0.0,3.0,2.0,31.0,0.0,1,0.0,29.0,14.0,0.559231,1.677693,0.666667,15.5,0.0,0.5,9.666667,7.0,4,"""Gunner Olszewski""","""WR""",2019,3,1.7575,0.585833,0.0025,false,0.003,2.812747,0.937582,0.004001,null,null,2021,2022,2.1,2,4.2,1.235,0.01,6.076081,3.03804,1.786657
"""00-0030569""","""S.Renfree""",2015,6.0,3.0,11.0,0.0,1.0,2.0,-0.824322,-7.4189,0.5,1.833333,0.0,0.166667,2.0,-4.0,0.0,0,0.0,-0.967829,-1.935659,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,"""Sean Renfree""","""QB""",2013,4,2.205896,0.551474,0.045896,false,0.004,5.401755,1.350439,0.112389,7,249,

In [0]:
dataset.filter(pl.col("player") == "Christian McCaffrey") \
       .sort("season")

player_id,player_name,season,pass_attempts,completions,pass_yards,pass_tds,interceptions,sacks,pass_epa_per_play,pass_total_epa,completion_pct,yards_per_attempt,pass_td_rate,int_rate,carries,rush_yards,rush_tds,explosive_runs,rush_fumbles,rush_epa_per_play,rush_total_epa,yards_per_carry,rush_td_rate,explosive_run_rate,fumble_rate,targets,receptions,rec_yards,rec_tds,explosive_recs,rec_fumbles,air_yards,yac,rec_epa_per_play,rec_total_epa,catch_rate,yards_per_rec,rec_td_rate,explosive_rec_rate,air_yards_per_target,yac_per_rec,games_played,player,position,curr_year_signed,curr_years,curr_value,curr_apy,curr_guaranteed,is_active,curr_apy_cap_pct,curr_inflated_value,curr_inflated_apy,curr_inflated_guaranteed,draft_round,draft_overall,year_end,next_year_signed,next_apy,next_years,next_value,next_guaranteed,next_apy_cap_pct,next_inflated_value,next_inflated_apy,next_inflated_guaranteed
str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,str,str,i32,i32,f64,f64,f64,bool,f64,f64,f64,f64,i32,i32,i32,i32,f64,i32,f64,f64,f64,f64,f64,f64
"""00-0033280""","""C.McCaffrey""",2017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117.0,435.0,2.0,8,1.0,-0.110231,-12.896998,3.717949,0.017094,0.068376,0.008547,113.0,80.0,651.0,5.0,7,0.0,245.0,593.0,0.100921,11.404118,0.707965,8.1375,0.044248,0.0875,2.168142,7.4125,16,"""Christian McCaffrey""","""RB""",2017,4,17.241304,4.310326,17.241304,false,0.026,31.096292,7.774073,31.096292,1,8,2020,2020,16.015853,4,64.063412,30.0625,0.081,97.3557,24.338925,45.685293
"""00-0033280""","""C.McCaffrey""",2018,1.0,1.0,50.0,1.0,0.0,0.0,6.549202,6.549202,1.0,50.0,1.0,0.0,219.0,1098.0,7.0,31,2.0,0.034037,7.454052,5.013699,0.031963,0.141553,0.009132,125.0,107.0,867.0,6.0,11,2.0,84.0,859.0,0.2668,33.349967,0.856,8.102804,0.048,0.102804,0.672,8.028037,16,"""Christian McCaffrey""","""RB""",2017,4,17.241304,4.310326,17.241304,false,0.026,31.096292,7.774073,31.096292,1,8,2020,2020,16.015853,4,64.063412,30.0625,0.081,97.3557,24.338925,45.685293
"""00-0033280""","""C.McCaffrey""",2019,2.0,0.0,0.0,0.0,0.0,0.0,-0.957607,-1.915215,0.0,0.0,0.0,0.0,288.0,1387.0,15.0,31,1.0,-0.049486,-14.251966,4.815972,0.052083,0.107639,0.003472,142.0,116.0,1005.0,4.0,7,0.0,111.0,987.0,0.355514,50.483007,0.816901,8.663793,0.028169,0.060345,0.78169,8.508621,16,"""Christian McCaffrey""","""RB""",2017,4,17.241304,4.310326,17.241304,false,0.026,31.096292,7.774073,31.096292,1,8,2020,2020,16.015853,4,64.063412,30.0625,0.081,97.3557,24.338925,45.685293
"""00-0033280""","""C.McCaffrey""",2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59.0,225.0,5.0,4,0.0,0.066672,3.933632,3.813559,0.084746,0.067797,0.0,19.0,17.0,146.0,1.0,1,0.0,27.0,120.0,0.469038,8.911715,0.894737,8.588235,0.052632,0.058824,1.421053,7.058824,3,"""Christian McCaffrey""","""RB""",2020,4,64.063412,16.015853,30.0625,false,0.081,97.3557,24.338925,45.685293,1,8,2023,2024,19.0,2,38.0,24.0,0.074,44.814409,22.407204,28.303837
"""00-0033280""","""C.McCaffrey""",2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.0,442.0,1.0,15,1.0,0.01796,1.778066,4.464646,0.010101,0.151515,0.010101,41.0,37.0,343.0,1.0,4,0.0,91.0,258.0,0.514229,21.0834,0.902439,9.27027,0.02439,0.108108,2.219512,6.972973,7,"""Christian McCaffrey""","""RB""",2020,4,64.063412,16.015853,30.0625,false,0.081,97.3557,24.338925,45.685293,1,8,2023,2024,19.0,2,38.0,24.0,0.074,44.814409,22.407204,28.303837
"""00-0033280""","""C.McCaffrey""",2022,1.0,1.0,34.0,1.0,0.0,0.0,3.771136,3.771136,1.0,34.0,1.0,0.0,246.0,1139.0,8.0,28,1.0,0.008135,2.001198,4.630081,0.03252,0.113821,0.004065,108.0,85.0,741.0,5.0,8,0.0,133.0,695.0,0.190477,20.57156,0.787037,8.717647,0.046296,0.094118,1.231481,8.176471,17,"""Christian McCaffrey""","""RB""",2020,4,64.063412,16.015853,30.0625,false,0.081,97.3557,24.338925,45.685293,1,8,2023,2024,19.0,2,38.0,24.0,0.074,44.814409,22.407204,28.303837
"""00-0033280""","""C.McCaffrey""",2023,0.0,0.0,0.